# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya – Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll navigate the dataset, inspect its record sets and fields by their `@id`s, extract records as DataFrames, perform basic exploratory data analysis (EDA), and visualize key features of the dataset.

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)
- The schema enforces referencing of all entities by their `@id`.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and initialize access to record sets with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

**Note:** All IDs are referenced by `@id` according to Croissant best practices.

In [ ]:
# Find all available record set IDs
print('Available record set @id values:')
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"  - {record_set['@id']}")
    record_set_ids.append(record_set['@id'])

# Explore fields available in each record set (by @id)
record_set_fields = {}
for rs_id in record_set_ids:
    rset = dataset.get_record_set(rs_id)
    field_ids = [field['@id'] for field in rset['field']]
    record_set_fields[rs_id] = field_ids
    print(f"\nFields in record set '{rs_id}':")
    for fid in field_ids:
        print(f"  - {fid}")

## 3. Data Extraction
Load data from each record set into separate pandas DataFrames for analysis. All references use `@id`.

In [ ]:
# Extract each record set as a DataFrame, keyed by record set @id
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record_set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  - Loaded {len(df)} rows, columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  - [Skipped or Empty] Could not load records for {rs_id}: {e}")

# Example: show the first five rows of the first record set, if available
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f'\nColumns of record set {example_rs_id}:')
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's inspect one of the main record sets. We'll:
- Select a numeric field (by its `@id`).
- Filter records where the numeric field is above a threshold.
- Normalize the numeric field.
- Optionally group by another categorical field.

All references strictly use `@id` identifiers for columns/fields.

In [ ]:
# Choose the first non-empty DataFrame for EDA
eda_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        eda_rs_id = rs_id
        break

if eda_rs_id:
    print(f'Using record_set @id for EDA: {eda_rs_id}')
    df_eda = dataframes[eda_rs_id]

    # List available fields
    print('Available fields (columns by @id):')
    print(df_eda.columns.tolist())

    # Attempt to select a numeric field (by guessing from dtype or column name)
    numeric_field_id = None
    for col in df_eda.columns:
        # Try to infer if this is a numeric field by dtype
        if pd.api.types.is_numeric_dtype(df_eda[col]):
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f'Selected numeric field for EDA: {numeric_field_id}')
        threshold = df_eda[numeric_field_id].quantile(0.75)  # Use 75th percentile as threshold

        filtered_df = df_eda[df_eda[numeric_field_id] > threshold].copy()
        print(f'Filtered records where {numeric_field_id} > {threshold:.2f}:')
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a likely categorical field (any non-numeric, if exists)
        group_field_id = None
        for col in df_eda.columns:
            if not pd.api.types.is_numeric_dtype(df_eda[col]):
                group_field_id = col
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f'Grouped data (mean {numeric_field_id}) by {group_field_id}:')
            display(grouped_df.head())
        else:
            print('No non-numeric field found for grouping.')
    else:
        print('No numeric field found for analysis in this record set.')
else:
    print('No non-empty record sets available for analysis.')

## 5. Visualization
Here's a visualization of the numeric field distribution (if found), using only `@id` to reference columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if eda_rs_id and numeric_field_id and not df_eda.empty:
    plt.figure(figsize=(8, 6))
    sns.histplot(df_eda[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
- This notebook demonstrated how to load and explore the FAIR² dataset using `mlcroissant`, referencing all data entities by their `@id`.
- We've listed available record sets, fields, loaded data for analysis, and performed simple EDA including normalization and grouping.
- For more advanced processing, consult field documentation in the Croissant schema and experiment with using additional record sets and fields by their `@id`s.